In [ ]:
# train_vit_psl_albu_v2.py
# Albumentations 2.0.8 compatible, robust to image shapes & channels, 80:20 stratified split.

import os, time, random
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from timm.loss import SoftTargetCrossEntropy
from timm.data import Mixup
from timm.scheduler import CosineLRScheduler
from timm.utils import ModelEmaV2  # <-- NativeScaler removed

# =======================
# CONFIG
# =======================
DATA_ROOT = "/home/gpu/PSL Isolated Signs Datasets/Combined-cropped-dataset"  # classes as subfolders
OUT_DIR   = "/home/gpu/PSL Isolated Signs Datasets"                           # where to save weights
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = 224
NUM_CLASSES = 39
VAL_SPLIT = 0.20
BATCH_SIZE = 60
EPOCHS = 200
EARLY_STOP_PATIENCE = 15
RANDOM_SEED = 42

MODEL_NAME = "deit3_base_patch16_224"   # robust data-efficient ViT
USE_HFLIP = True                        # set False if left/right hand matters

BASE_LR = 5e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTH = 0.1
MIXUP_ALPHA = 0.8
CUTMIX_ALPHA = 1.0
MIXUP_PROB = 1.0
MIXUP_SWITCH_PROB = 0.5
WARMUP_EPOCHS = 5
GRAD_CLIP_NORM = 1.0

# =======================
# UTILS
# =======================
def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def safe_imread_rgb(path: str):
    """Robust image loader: handles grayscale, RGB, RGBA; returns RGB uint8."""
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f"Failed to read image: {path}")
    # Handle channel cases
    if img.ndim == 2:
        # grayscale -> RGB
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            # BGRA -> BGR then to RGB (drop alpha)
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        elif img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            # unexpected channel count -> convert by first 3 channels
            img = img[:, :, :3]
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        # Unexpected shape -> try force decode as color
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Failed to re-read image as color: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root=root)
        self.atransform = transform

    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

def make_train_tfms(img_size=IMG_SIZE, use_hflip=USE_HFLIP):
    return A.Compose([
        # v2 API: size=(H,W)
        A.RandomResizedCrop(size=(img_size, img_size),
                            scale=(0.6, 1.0), ratio=(0.8, 1.25),
                            interpolation=cv2.INTER_CUBIC),
        A.HorizontalFlip(p=0.5 if use_hflip else 0.0),

        # Geometric (keep hands plausible)
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=30,
                           border_mode=cv2.BORDER_REFLECT_101, p=0.9),
        A.Perspective(scale=(0.02, 0.07), keep_size=True,
                      pad_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.GridDistortion(num_steps=5, distort_limit=0.2,
                         border_mode=cv2.BORDER_REFLECT_101, p=0.2),

        # Color/contrast
        A.OneOf([
            A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.35, hue=0.04),
            A.RandomGamma(gamma_limit=(70, 130)),
            A.CLAHE(clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)),
        ], p=0.9),

        # Blur & camera artifacts
        A.OneOf([
            A.MotionBlur(blur_limit=7),
            A.GaussianBlur(blur_limit=(3, 7)),
            A.MedianBlur(blur_limit=5),
        ], p=0.5),
        A.ImageCompression(quality_lower=40, quality_upper=90, p=0.5),
        A.GaussNoise(var_limit=(10.0, 80.0), p=0.4),

        # Lighting & shadows
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.RandomShadow(num_shadows_lower=1, num_shadows_upper=2, shadow_dimension=5, p=0.3),

        # Occlusion
        A.CoarseDropout(max_holes=6,
                        max_height=int(img_size*0.18), max_width=int(img_size*0.18),
                        min_holes=1,
                        min_height=int(img_size*0.06), min_width=int(img_size*0.06),
                        fill_value=0, p=0.7),

        # Finalize
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

def make_val_tfms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size, interpolation=cv2.INTER_CUBIC),
        A.PadIfNeeded(min_height=img_size, min_width=img_size,
                      border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

def stratified_split_indices(samples, n_classes, val_ratio=VAL_SPLIT, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    by_class = defaultdict(list)
    for idx, (_, y) in enumerate(samples):
        by_class[y].append(idx)

    train_idx, val_idx = [], []
    for y in range(n_classes):
        ids = by_class[y]
        if not ids:
            continue
        ids = np.array(ids)
        rng.shuffle(ids)
        k = max(1, int(round(val_ratio * len(ids))))
        v_ids = ids[:k]
        t_ids = ids[k:]
        if len(t_ids) == 0:   # if class has only one sample
            t_ids, v_ids = ids, ids[:0]
        train_idx.extend(t_ids.tolist())
        val_idx.extend(v_ids.tolist())

    rng.shuffle(train_idx); rng.shuffle(val_idx)
    return np.array(train_idx), np.array(val_idx)

def build_loaders():
    # Build two views (train/val pipelines), but split on the SAME ordering
    train_view = AlbumentationsImageFolder(DATA_ROOT, transform=make_train_tfms())
    val_view   = AlbumentationsImageFolder(DATA_ROOT, transform=make_val_tfms())

    n_classes = len(train_view.classes)
    assert n_classes == NUM_CLASSES, f"Found {n_classes} classes, but NUM_CLASSES={NUM_CLASSES}"

    tr_idx, va_idx = stratified_split_indices(train_view.samples, n_classes, VAL_SPLIT, RANDOM_SEED)

    train_ds = Subset(train_view, tr_idx)
    val_ds   = Subset(val_view, va_idx)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=8, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=8, pin_memory=True)
    return train_loader, val_loader, n_classes

def build_model(num_classes):
    model = timm.create_model(
        MODEL_NAME, pretrained=True, num_classes=num_classes,
        drop_rate=0.1, drop_path_rate=0.1  # dropout + stochastic depth
    )
    return model

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    loss_meter, correct, total = 0.0, 0, 0
    crit = nn.CrossEntropyLoss()
    use_amp = torch.cuda.is_available()

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if use_amp:
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = crit(logits, targets)
        else:
            logits = model(images)
            loss = crit(logits, targets)

        loss_meter += loss.item() * images.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == targets).sum().item()
        total += images.size(0)

    return (correct / total * 100.0), (loss_meter / total)

def main():
    print("Albumentations version:", A.__version__)
    set_seed()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_loader, val_loader, n_classes = build_loaders()
    print(f"Classes: {n_classes} | Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    model = build_model(n_classes).to(device)
    model_ema = ModelEmaV2(model, decay=0.9999, device=device)

    mixup_fn = Mixup(
        mixup_alpha=MIXUP_ALPHA, cutmix_alpha=CUTMIX_ALPHA,
        prob=MIXUP_PROB, switch_prob=MIXUP_SWITCH_PROB, mode='batch',
        label_smoothing=LABEL_SMOOTH, num_classes=n_classes
    )

    criterion = SoftTargetCrossEntropy()
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())  # <-- FIX

    total_steps = EPOCHS * len(train_loader)
    warmup_steps = max(1, int(WARMUP_EPOCHS * len(train_loader)))
    scheduler = CosineLRScheduler(
        optimizer,
        t_initial=total_steps - warmup_steps,
        lr_min=1e-6,
        warmup_t=warmup_steps,
        warmup_lr_init=1e-6,
        k_decay=1.0
    )

    best_acc = 0.0
    best_val_loss = float("inf")
    steps = 0
    epochs_no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0

        for images, targets in train_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            # Mixup/CutMix -> soft labels
            images, targets = mixup_fn(images, targets)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, targets)

            # GradScaler path
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)  # so clip sees true grads
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()

            scheduler.step(steps)
            steps += 1
            model_ema.update(model)

            running_loss += loss.item() * images.size(0)
            # accuracy proxy with soft labels: compare argmaxes
            preds = logits.softmax(dim=1).argmax(dim=1)
            hards = targets.argmax(dim=1)
            running_correct += (preds == hards).sum().item()
            running_total += images.size(0)

        train_loss = running_loss / running_total
        train_acc = 100.0 * running_correct / running_total

        val_acc, val_loss = evaluate(model_ema.module, val_loader, device)

        print(f"Epoch {epoch:03d}/{EPOCHS} | "
              f"train_loss {train_loss:.4f} acc {train_acc:.2f}% | "
              f"val_loss {val_loss:.4f} acc {val_acc:.2f}% | "
              f"lr {optimizer.param_groups[0]['lr']:.2e}")

        # Save best by accuracy
        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(OUT_DIR, "best_vit_psl.pth")
            torch.save({
                "model": model_ema.module.state_dict(),
                "acc": best_acc,
                "epoch": epoch,
                "classes": n_classes,
                "model_name": MODEL_NAME,
                "img_size": IMG_SIZE
            }, save_path)
            print(f"  ↳ Saved new BEST by acc: {best_acc:.2f}% → {save_path}")

        # Early stopping on val loss
        if val_loss + 1e-4 < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"Early stopping (no val loss improvement for {EARLY_STOP_PATIENCE} epochs).")
                break

    print(f"Training done. Best val acc: {best_acc:.2f}% | Best val loss: {best_val_loss:.4f}")
    print(f"Saved weights at: {os.path.join(OUT_DIR, 'best_vit_psl.pth')}")

if __name__ == "__main__":
    main()


In [ ]:
# All-in-one TTA eval & inference (Albumentations 2.0.8)

import os, csv, argparse
from pathlib import Path
from typing import List, Tuple

import cv2, numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---------- Utils ----------
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: 
        raise RuntimeError(f"Failed to read: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def load_checkpoint(weights_path: str):
    ckpt = torch.load(weights_path, map_location="cpu")
    model_name = ckpt.get("model_name", "deit3_base_patch16_224")
    num_classes = ckpt.get("classes", 39)
    img_size = ckpt.get("img_size", 224)
    state_dict = ckpt["model"]
    return model_name, num_classes, img_size, state_dict

def build_model(model_name: str, num_classes: int, state_dict: dict, device: str):
    model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()
    return model

# ---------- TTA builders ----------
def make_single_tta(img_size: int, scale: float, angle: float, do_flip: bool):
    """Deterministic val-like preprocessing with scale/rotate/flip variants."""
    ops = []
    # Resize longest side, then center-crop (if upscaled) or pad (if downscaled)
    if scale >= 1.0:
        max_side = max(1, int(round(img_size * scale)))
        ops.append(A.LongestMaxSize(max_size=max_side, interpolation=cv2.INTER_CUBIC))
        ops.append(A.CenterCrop(img_size, img_size))
    else:
        max_side = max(1, int(round(img_size * scale)))
        ops.append(A.LongestMaxSize(max_size=max_side, interpolation=cv2.INTER_CUBIC))
        ops.append(A.PadIfNeeded(min_height=img_size, min_width=img_size,
                                 border_mode=cv2.BORDER_REFLECT_101))
    if abs(angle) > 0:
        # keep output size; reflect borders to avoid black corners
        ops.append(A.Affine(rotate=angle, fit_output=False,
                            border_mode=cv2.BORDER_REFLECT_101))
    if do_flip:
        ops.append(A.HorizontalFlip(p=1.0))
    ops += [
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
    return A.Compose(ops)

def make_tta_list(img_size: int, scales: List[float], angles: List[float], flip: bool):
    tfs = []
    for s in scales:
        for a in angles:
            tfs.append(make_single_tta(img_size, s, a, False))
            if flip:
                tfs.append(make_single_tta(img_size, s, a, True))
    return tfs

# ---------- Datasets (raw RGB -> transform per TTA pass) ----------
class AlbumentationsImageFolderRaw(ImageFolder):
    def __init__(self, root):
        super().__init__(root=root)
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        return img, target

class ImageListRaw(Dataset):
    def __init__(self, files: List[str]):
        self.files = files
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        p = self.files[idx]
        img = safe_imread_rgb(p)
        return img, p

def list_images(path: str) -> List[str]:
    exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
    p = Path(path)
    if p.is_file(): return [str(p)]
    return sorted([str(f) for f in p.rglob("*") if f.suffix.lower() in exts])

# ---------- Core TTA loops ----------
@torch.no_grad()
def eval_with_tta(data_root: str, weights: str, out_dir: str,
                  scales=[0.9,1.0,1.1], angles=[-8,0,8], flip=False,
                  bs=64, workers=4):
    """Evaluate accuracy with logits-averaged TTA on an ImageFolder (labeled)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name, n_cls, img_size, state = load_checkpoint(weights)
    model = build_model(model_name, n_cls, state, device)

    base_ds = AlbumentationsImageFolderRaw(data_root)
    classes = base_ds.classes
    N = len(base_ds)

    tta_list = make_tta_list(img_size, scales, angles, flip)
    sum_logits = torch.zeros((N, n_cls), dtype=torch.float32, device=device)

    use_amp = torch.cuda.is_available()
    idx_offset = 0
    for t_idx, tfm in enumerate(tta_list, 1):
        class _View(Dataset):
            def __init__(self, base, tfm): self.base, self.tfm = base, tfm
            def __len__(self): return len(self.base)
            def __getitem__(self, i):
                img, y = self.base[i]
                x = self.tfm(image=img)["image"]
                return x, y
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"TTA pass {t_idx}/{len(tta_list)} complete.")

    preds = sum_logits.argmax(dim=1).cpu().numpy()
    targets = np.array([y for _, y in base_ds.samples], dtype=np.int64)
    acc = (preds == targets).mean() * 100.0

    # confusion matrix
    cm = np.zeros((n_cls, n_cls), dtype=np.int64)
    for t, p in zip(targets, preds):
        cm[t, p] += 1

    os.makedirs(out_dir, exist_ok=True)
    # summary
    with open(os.path.join(out_dir, "eval_tta_summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"Model: {model_name}\nNum classes: {n_cls}\nImg size: {img_size}\n")
        f.write(f"TTA: scales={scales}, angles={angles}, flip={flip}\n")
        f.write(f"Accuracy: {acc:.2f}%\n")

    # per-class accuracy
    with open(os.path.join(out_dir, "per_class_acc.txt"), "w", encoding="utf-8") as f:
        for i, name in enumerate(classes):
            total_i = cm[i].sum()
            pc = 100.0 * (cm[i,i] / total_i) if total_i > 0 else 0.0
            f.write(f"{name}: {pc:.2f}\n")

    # confusion matrix CSV
    with open(os.path.join(out_dir, "confusion_matrix_tta.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow([""] + classes)
        for i, row in enumerate(cm): w.writerow([classes[i]] + row.tolist())

    print(f"TTA eval done. Acc: {acc:.2f}%  →  {out_dir}")
    return acc

@torch.no_grad()
def infer_with_tta(data_path: str, weights: str, out_csv: str,
                   label_names_root="", scales=[0.9,1.0,1.1], angles=[-8,0,8], flip=False,
                   bs=64, workers=4):
    """Infer on unlabeled images; save CSV with top1 + prob (TTA averaged)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name, n_cls, img_size, state = load_checkpoint(weights)
    model = build_model(model_name, n_cls, state, device)

    files = list_images(data_path)
    if not files:
        print(f"No images at {data_path}"); 
        return

    # class names (optional)
    if label_names_root and Path(label_names_root).is_dir():
        tmp = ImageFolder(label_names_root)
        class_names = tmp.classes
    else:
        class_names = [f"class_{i}" for i in range(n_cls)]

    base_ds = ImageListRaw(files)
    N = len(base_ds)
    tta_list = make_tta_list(img_size, scales, angles, flip)
    sum_logits = torch.zeros((N, n_cls), dtype=torch.float32, device=device)

    use_amp = torch.cuda.is_available()
    for t_idx, tfm in enumerate(tta_list, 1):
        class _View(Dataset):
            def __init__(self, base, tfm): self.base, self.tfm = base, tfm
            def __len__(self): return len(self.base)
            def __getitem__(self, i):
                img, p = self.base[i]
                x = self.tfm(image=img)["image"]
                return x, p
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, paths in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"TTA pass {t_idx}/{len(tta_list)} complete.")

    probs = sum_logits.softmax(dim=1).cpu().numpy()
    top1_idx = probs.argmax(axis=1)
    top1_prob = probs.max(axis=1)

    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["path","top1","top1_prob"])
        for p, cls_i, pr in zip(files, top1_idx, top1_prob):
            w.writerow([p, class_names[int(cls_i)], f"{pr:.4f}"])
    print(f"Inference (TTA) saved → {out_csv}")


In [ ]:
# Paths from your training run
WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/best_vit_psl.pth"
LABELED_DATA = "/home/gpu/PSL-Cropped/val"
OUT_DIR = "/home/gpu/PSL Isolated Signs Datasets/eval_tta_out"

# If left/right hand is NOT class-specific, set flip=True for a small boost
FLIP_SAFE = False   # ← change to True if safe for your labels

# Highest-accuracy evaluation with TTA
acc = eval_with_tta(
    data_root=LABELED_DATA,
    weights=WEIGHTS,
    out_dir=OUT_DIR,
    scales=[0.9, 1.0, 1.1],
    angles=[-8, 0, 8],
    flip=FLIP_SAFE,
    bs=64,
    workers=4
)
print("Final TTA Accuracy:", f"{acc:.2f}%")


In [ ]:
import os, torch, timm
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/best_vit_psl.pth"
print("Weights exists:", os.path.exists(WEIGHTS))

ckpt = torch.load(WEIGHTS, map_location="cpu")
print("Checkpoint keys:", list(ckpt.keys()))
print("Model in ckpt:", ckpt.get("model_name", "deit3_base_patch16_224"))
print("Classes in ckpt:", ckpt.get("classes", None), "Img size:", ckpt.get("img_size", None))


In [ ]:
# train_deit3_base_patch16_224_no_ckpt.py
# - Model: deit3_base_patch16_224 (timm)
# - Train/Val from separate dirs (no stratified split)
# - Batch size fixed with gradient accumulation
# - Accurate metrics: Acc, Precision (macro), Recall (macro), F1 (macro), Confusion Matrix, Top-1, Top-5
# - AMP shim (torch 1.x / 2.x), EMA, CosineLRScheduler (timm) w/ step_update()
# - Strict validations (classes, counts, shapes, logits, params)
# - HARD block of activation checkpointing (Torch + timm) to avoid CheckpointError

import os, random
from contextlib import contextmanager

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from timm.loss import SoftTargetCrossEntropy
from timm.data import Mixup, resolve_model_data_config
from timm.scheduler import CosineLRScheduler
from timm.utils import ModelEmaV2

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ==================== CONFIG ====================
TRAIN_DIR = "/home/gpu/PSL Isolated Signs Datasets/UAlpha-CFRT-combined-dataset"
VAL_DIR   = "/home/gpu/PSL Isolated Signs Datasets/PSL-Dataset-2021/val"
OUT_DIR   = "/home/gpu/PSL Isolated Signs Datasets/PSL-Images-2021-Dataset-out"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME  = "deit3_base_patch16_224"
NUM_CLASSES = 39

EPOCHS        = 100
FREEZE_EPOCHS = 3  # freeze backbone for warmup (linear probe), then unfreeze

BATCH_SIZE  = 100     # you can set 350 if VRAM allows
ACCUM_STEPS = 2       # effective batch = BATCH_SIZE * ACCUM_STEPS

# Aug/Reg (standard for DeiT training)
LABEL_SMOOTH      = 0.1
MIXUP_ALPHA       = 0.8
CUTMIX_ALPHA      = 1.0
MIXUP_PROB        = 1.0
MIXUP_SWITCH_PROB = 0.5

DROP_RATE      = 0.0
DROP_PATH_RATE = 0.25  # typical DeiT3 base can use 0.1–0.25; keep as before

# Optim & LR
BASE_LR_FULL   = 5e-5
BASE_LR_LINEAR = 1e-3
WEIGHT_DECAY   = 0.05
WARMUP_EPOCHS  = 5
GRAD_CLIP_NORM = 1.0

EARLY_STOP_PATIENCE = 15
RANDOM_SEED         = 42

torch.backends.cudnn.benchmark = True

# ==================== AMP SHIM (torch 1.x / 2.x) ====================
USE_TORCH_AMP = hasattr(torch, "amp") and hasattr(torch.amp, "autocast")
if USE_TORCH_AMP:
    def autocast_ctx():
        return torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available())
    GradScaler = torch.amp.GradScaler
else:
    from torch.cuda.amp import autocast as _autocast_cuda
    from torch.cuda.amp import GradScaler as _GradScaler
    def autocast_ctx():
        return _autocast_cuda(enabled=torch.cuda.is_available())
    GradScaler = _GradScaler

# ==================== HARD DISABLE CHECKPOINTING (Torch + timm) ====================
def _make_no_ckpt_fn():
    def _no_ckpt(function, *args, **kwargs):
        # Ignore preserve_rng_state/use_reentrant etc.; just call
        return function(*args)
    return _no_ckpt

def _make_no_ckpt_seq_fn():
    # Replacement for checkpoint_seq: run sequentially without checkpointing
    def _no_ckpt_seq(functions, *args, **kwargs):
        x = args[0] if len(args) > 0 else None
        for fn in functions:
            x = fn(x)
        return x
    return _no_ckpt_seq

@contextmanager
def disable_all_checkpointing():
    """
    Temporarily replace ALL known checkpoint entry points:
      - torch.utils.checkpoint.checkpoint
      - timm.models.layers.checkpoint / checkpoint_seq (older timm)
      - timm.layers.checkpoint / checkpoint_seq (newer timm)
    """
    patches = []

    # 1) Torch
    try:
        import torch.utils.checkpoint as _cp
        if hasattr(_cp, "checkpoint"):
            patches.append((_cp, "checkpoint", _cp.checkpoint, _make_no_ckpt_fn()))
    except Exception:
        pass

    # 2) timm.models.layers (older timm)
    try:
        import timm.models.layers as tml
        if hasattr(tml, "checkpoint"):
            patches.append((tml, "checkpoint", tml.checkpoint, _make_no_ckpt_fn()))
        if hasattr(tml, "checkpoint_seq"):
            patches.append((tml, "checkpoint_seq", tml.checkpoint_seq, _make_no_ckpt_seq_fn()))
    except Exception:
        pass

    # 3) timm.layers (newer timm)
    try:
        import timm.layers as tl
        if hasattr(tl, "checkpoint"):
            patches.append((tl, "checkpoint", tl.checkpoint, _make_no_ckpt_fn()))
        if hasattr(tl, "checkpoint_seq"):
            patches.append((tl, "checkpoint_seq", tl.checkpoint_seq, _make_no_ckpt_seq_fn()))
    except Exception:
        pass

    # Apply patches
    for mod, name, orig, repl in patches:
        try:
            setattr(mod, name, repl if callable(repl) else repl())
        except Exception:
            pass

    try:
        yield
    finally:
        # Restore originals
        for mod, name, orig, repl in patches[::-1]:
            try:
                setattr(mod, name, orig)
            except Exception:
                pass

# ==================== UTILS ====================
def set_seed(seed=RANDOM_SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root=root); self.atransform = transform
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

def make_train_tfms(img_size, mean, std):
    # DeiT3 expects 224x224; use RandomResizedCrop to that size + light aug
    return A.Compose([
        A.RandomResizedCrop(height=img_size, width=img_size, scale=(0.55, 1.0), ratio=(0.8, 1.25)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

def make_val_tfms(img_size, mean, std):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

def make_loader(ds, batch_size, shuffle, workers=8):
    return DataLoader(
        dataset=ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=workers, pin_memory=True, drop_last=shuffle, persistent_workers=True
    )

def build_model(num_classes):
    # Create DeiT3-B/16—returns logits [B, num_classes]
    model = timm.create_model(
        model_name=MODEL_NAME,
        pretrained=True,
        num_classes=num_classes,
        drop_rate=DROP_RATE,
        drop_path_rate=DROP_PATH_RATE
    )
    # Explicitly ensure any grad-checkpointing toggles are OFF
    if hasattr(model, "set_grad_checkpointing"):
        try:
            model.set_grad_checkpointing(enable=False)
        except TypeError:
            pass
    if hasattr(model, "grad_checkpointing"):
        try:
            model.grad_checkpointing = False
        except Exception:
            pass
    return model

def set_backbone_trainable(m: nn.Module, trainable: bool):
    # DeiT/ViT heads are usually named 'head' (compatible with this set)
    head_names = ['head', 'fc', 'classifier', 'heads']
    for n, p in m.named_parameters():
        p.requires_grad = trainable or any(h in n for h in head_names)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    model.eval()
    crit = nn.CrossEntropyLoss()
    y_true, y_pred = [], []
    loss_meter, total = 0.0, 0
    top5_correct = 0
    k = min(5, num_classes)

    for images, targets in loader:
        assert images.ndim == 4, "Images must be NCHW [B,C,H,W]"
        assert targets.ndim == 1, "Targets must be 1D class indices"

        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with disable_all_checkpointing():
            with autocast_ctx():
                logits = model(images)
                loss = crit(logits, targets)

        assert logits.ndim == 2 and logits.shape[1] == num_classes, \
            f"Logits must be [B, {num_classes}] but got {tuple(logits.shape)}"
        assert targets.max().item() < num_classes, "Target index exceeds num_classes"

        loss_meter += loss.item() * images.size(0)
        total += images.size(0)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)
        y_true.extend(targets.detach().cpu().numpy())
        y_pred .extend(preds.detach().cpu().numpy())

        # Top-k (Top-5 or Top-k=min(5,num_classes))
        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1, 1)).any(dim=1).sum().item()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    top1 = acc * 100.0
    top5 = 100.0 * (top5_correct / max(1, total))

    return {
        "acc": acc * 100.0,
        "prec": prec * 100.0,
        "rec": rec * 100.0,
        "f1": f1 * 100.0,
        "cm": cm,
        "loss": loss_meter / max(1, total),
        "top1": top1,
        "top5": top5
    }

# ==================== MAIN ====================
def main():
    print("CUDA:", torch.version.cuda if torch.cuda.is_available() else "none",
          "| PyTorch:", torch.__version__)
    set_seed()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Model & data config (DeiT3 expects 224x224; resolve_config returns it)
    model = build_model(num_classes=NUM_CLASSES).to(device)
    cfg = resolve_model_data_config(model)
    img_size = cfg["input_size"][-1]; mean, std = cfg["mean"], cfg["std"]

    # Datasets from separate dirs
    train_ds = AlbumentationsImageFolder(root=TRAIN_DIR, transform=make_train_tfms(img_size, mean, std))
    val_ds   = AlbumentationsImageFolder(root=VAL_DIR,   transform=make_val_tfms(img_size, mean, std))

    # ---- Cross-checks ----
    assert len(train_ds.classes) == NUM_CLASSES, f"Train classes={len(train_ds.classes)} != NUM_CLASSES={NUM_CLASSES}"
    assert train_ds.classes == val_ds.classes, "Class order/names differ between train and val"
    print(f"✔ Classes: {NUM_CLASSES}")
    print(f"✔ Train imgs: {len(train_ds)} | Val imgs: {len(val_ds)}")
    train_counts = [0]*NUM_CLASSES; val_counts = [0]*NUM_CLASSES
    for _, y in train_ds.samples: train_counts[y] += 1
    for _, y in val_ds.samples:   val_counts[y]   += 1
    for i, cls in enumerate(train_ds.classes):
        print(f"  {i:02d}: {cls} | Train {train_counts[i]:5d} | Val {val_counts[i]:5d}")
    assert sum(train_counts) == len(train_ds) and sum(val_counts) == len(val_ds), "Per-class counts mismatch sums"

    # DataLoaders
    train_loader = make_loader(ds=train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = make_loader(ds=val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    # EMA, Mixup, Loss
    model_ema = ModelEmaV2(model=model, decay=0.9999, device=device)
    mixup_fn = Mixup(
        mixup_alpha=MIXUP_ALPHA,
        cutmix_alpha=CUTMIX_ALPHA,
        prob=MIXUP_PROB,
        switch_prob=MIXUP_SWITCH_PROB,
        mode="batch",
        correct_lam=True,
        label_smoothing=LABEL_SMOOTH,
        num_classes=NUM_CLASSES
    )
    criterion = SoftTargetCrossEntropy()

    # Freeze backbone initially (optimizer includes ALL params; unfreeze needs no rebuild)
    set_backbone_trainable(m=model, trainable=False if FREEZE_EPOCHS > 0 else True)
    total_params, trainable_params = count_params(model)
    print(f"✔ Model {MODEL_NAME} | Params total={total_params:,} | trainable={trainable_params:,}")

    optimizer = torch.optim.AdamW(
        params=model.parameters(),  # include all params
        lr=BASE_LR_LINEAR if FREEZE_EPOCHS > 0 else BASE_LR_FULL,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )
    scaler = GradScaler(enabled=torch.cuda.is_available())

    steps_per_epoch = len(train_loader)
    total_steps     = EPOCHS * max(1, steps_per_epoch)
    warmup_steps    = max(1, int(WARMUP_EPOCHS * max(1, steps_per_epoch)))
    scheduler = CosineLRScheduler(
        optimizer=optimizer,
        t_initial=total_steps - warmup_steps,
        lr_min=1e-6,
        warmup_t=warmup_steps,
        warmup_lr_init=1e-6
    )

    best_acc = 0.0
    best_val_loss = float("inf")
    epochs_no_improve = 0
    steps = 0  # scheduler updates after each optimizer step (post-accum)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)

        for i, (images, targets_hard) in enumerate(train_loader):
            assert isinstance(images, torch.Tensor) and isinstance(targets_hard, torch.Tensor)
            assert images.ndim == 4 and images.size(0) <= BATCH_SIZE, "Unexpected batch shape/size"
            assert targets_hard.ndim == 1, "Targets must be 1D indices"

            images = images.to(device, non_blocking=True)
            targets_hard = targets_hard.to(device, non_blocking=True)

            # Preserve hard labels for accuracy; mixup returns soft labels
            hard_for_acc = targets_hard.clone()
            images, targets_soft = mixup_fn(images, targets_hard)

            with disable_all_checkpointing():
                with autocast_ctx():
                    logits = model(images)
                    assert logits.shape[1] == NUM_CLASSES, f"Logits dim {logits.shape[1]} != NUM_CLASSES {NUM_CLASSES}"
                    loss = criterion(logits, targets_soft)

            # Backward with grad accumulation
            scaler.scale(loss).backward()
            if (i + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step_update(num_updates=steps)
                steps += 1
                model_ema.update(model)

            # Bookkeeping
            running_loss += loss.item() * images.size(0)
            preds = logits.softmax(1).argmax(1)
            running_correct += (preds == hard_for_acc).sum().item()
            running_total += images.size(0)

        # Flush final partial accumulation step if needed
        if (len(train_loader) % ACCUM_STEPS) != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step_update(num_updates=steps)
            steps += 1
            model_ema.update(model)

        train_loss = running_loss / max(1, running_total)
        train_acc  = 100.0 * running_correct / max(1, running_total)

        # Unfreeze backbone after FREEZE_EPOCHS
        if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS:
            set_backbone_trainable(m=model, trainable=True)
            for g in optimizer.param_groups:
                g['lr'] = BASE_LR_FULL
            total_params, trainable_params = count_params(model)
            print(f"→ Unfroze backbone; LR -> {BASE_LR_FULL:g} | trainable params={trainable_params:,}")

        # ===== Validation (EMA model) =====
        metrics = evaluate(model=model_ema.module, loader=val_loader, device=device, num_classes=NUM_CLASSES)
        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"train loss {train_loss:.4f} acc {train_acc:.2f}% | "
            f"val loss {metrics['loss']:.4f} acc {metrics['acc']:.2f}% | "
            f"Prec {metrics['prec']:.2f}% | Rec {metrics['rec']:.2f}% | "
            f"F1 {metrics['f1']:.2f}% | Top1 {metrics['top1']:.2f}% | Top5 {metrics['top5']:.2f}%"
        )

        # Save best by accuracy
        if metrics["acc"] > best_acc:
            best_acc = metrics["acc"]
            save_path = os.path.join(OUT_DIR, "best_deit3_CFRT-Dataset-out.pth")
            torch.save({
                "model": model_ema.module.state_dict(),
                "acc": best_acc,
                "epoch": epoch,
                "classes": NUM_CLASSES,
                "class_names": train_ds.classes,
                "model_name": MODEL_NAME,
                "img_size": img_size,
                "cfg": cfg,
                "conf_matrix": metrics["cm"]
            }, save_path)
            print(f"  ↳ Saved new BEST acc: {best_acc:.2f}% → {save_path}")

        # Early stopping on validation loss
        if metrics["loss"] + 1e-4 < best_val_loss:
            best_val_loss = metrics["loss"]; epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"Early stopping (no val loss improvement for {EARLY_STOP_PATIENCE} epochs).")
                break

    print(f"Done. Best val acc: {best_acc:.2f}%")

if __name__ == "__main__":
    main()


/home/gpu/.local/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


CUDA: 12.4 | PyTorch: 2.5.1+cu124
✔ Classes: 39
✔ Train imgs: 25052 | Val imgs: 317
  00: ء | Train   650 | Val     8
  01: ا | Train   756 | Val    10
  02: ب | Train   761 | Val     8
  03: ت | Train   747 | Val    10
  04: ث | Train   708 | Val    10
  05: ج | Train    90 | Val     8
  06: ح | Train   716 | Val     8
  07: خ | Train   703 | Val     8
  08: د | Train   633 | Val     8
  09: ذ | Train   697 | Val     8
  10: ر | Train   751 | Val     8
  11: ز | Train   792 | Val     8
  12: س | Train   739 | Val     8
  13: ش | Train   727 | Val     8
  14: ص | Train   756 | Val     8
  15: ض | Train   668 | Val     8
  16: ط | Train   663 | Val     8
  17: ظ | Train   670 | Val     8
  18: ع | Train   629 | Val     8
  19: غ | Train   671 | Val     8
  20: ف | Train   635 | Val     8
  21: ق | Train   671 | Val     8
  22: ل | Train   677 | Val     8
  23: م | Train   647 | Val     8
  24: ن | Train   643 | Val     8
  25: و | Train   648 | Val     8
  26: ٹ | Train   699 | Val    1

/home/gpu/.conda/envs/python305/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Epoch 001/100 | train loss 3.7522 acc 3.41% | val loss 3.8318 acc 4.73% | Prec 0.57% | Rec 4.81% | F1 1.01% | Top1 4.73% | Top5 13.25%
  ↳ Saved new BEST acc: 4.73% → /home/gpu/PSL Isolated Signs Datasets/PSL-Images-2021-Dataset-out/best_deit3_CFRT-Dataset-out.pth
Epoch 002/100 | train loss 3.6511 acc 4.61% | val loss 3.8302 acc 4.73% | Prec 0.56% | Rec 4.81% | F1 1.01% | Top1 4.73% | Top5 13.25%
→ Unfroze backbone; LR -> 5e-05 | trainable params=85,846,311
Epoch 003/100 | train loss 3.5896 acc 5.77% | val loss 3.8284 acc 4.73% | Prec 0.56% | Rec 4.81% | F1 1.01% | Top1 4.73% | Top5 13.25%
